In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm import tqdm
from torch.utils.data import DataLoader

# GPU 설정
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("device: ", device)

device:  cuda:0


In [ ]:
df = pd.read_excel('/content/sample_data/Ai-hub_선정적&비선정적.xlsx')
df = df.drop_duplicates(subset=['origin_text'])
df = df[['origin_text', 'is_sexual']]
df['label'] = df['is_sexual']
df = df[['origin_text', 'label']]
df.dropna(inplace=True)

<ipython-input-3-3e7191fd2f8d>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [ ]:
train_data = df.sample(frac=0.8, random_state=42)
test_data = df.drop(train_data.index)

In [ ]:
MODEL_NAME = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# 토큰화
tokenized_train_sentences = tokenizer(
    list(train_data['origin_text']),
    return_tensors="pt",
    max_length=128,
    padding=True,
    truncation=True,
    add_special_tokens=True
)

In [ ]:
tokenized_test_sentences = tokenizer(
    list(test_data['origin_text']),
    return_tensors="pt",
    max_length=128,
    padding=True,
    truncation=True,
    add_special_tokens=True
)

In [ ]:
# 데이터셋 클래스 정의
class CurseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_label = train_data["label"].values
test_label = test_data["label"].values

train_dataset = CurseDataset(tokenized_train_sentences, train_label)
test_dataset = CurseDataset(tokenized_test_sentences, test_label)

In [ ]:
# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# 모델 정의 (dropout 추가)
class CustomModel(AutoModelForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.dropout = torch.nn.Dropout(0.5)  # dropout 비율 설정

model = CustomModel.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [ ]:
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir='./',
    num_train_epochs=5,
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    save_steps=500,  # save_steps를 eval_steps와 일치시킴
    eval_steps=500,   # eval_steps 추가
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model="pr-auc",
    greater_is_better=True,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# EarlyStoppingCallback 설정
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=2)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score, precision_recall_curve, auc

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)  # 예측된 클래스 (0 또는 1)
    preds_prob = pred.predictions[:, 1]  # 긍정 클래스(1)에 대한 확률값

    # Precision, Recall, F1, Accuracy 계산
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    # AUC-ROC 계산
    auc_roc = roc_auc_score(labels, preds_prob)

    # PR-AUC 계산
    precision_vals, recall_vals, _ = precision_recall_curve(labels, preds_prob)
    pr_auc = auc(recall_vals, precision_vals)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc-roc': auc_roc,
        'pr-auc': pr_auc
    }

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 클래스 가중치 계산
class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=train_label)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# 커스텀 Trainer 클래스 정의
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # 손실 함수에 클래스 가중치 적용
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# CustomTrainer 설정
trainer = CustomTrainer(
    model=model,  # 모델 객체
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]  # 콜백에 추가
)

In [ ]:
# 학습 시작
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc-roc,Pr-auc
500,0.694400,0.679170,0.945587,0.010956,0.120603,0.005739,0.576326,0.073717
1000,0.667800,0.583604,0.812274,0.284223,0.177693,0.709708,0.835697,0.312939
1500,0.510600,0.386305,0.941142,0.531909,0.456697,0.636777,0.902814,0.513327
2000,0.438100,0.335656,0.941355,0.555746,0.461453,0.698470,0.920761,0.556848
2500,0.430600,0.337179,0.947094,0.586434,0.497419,0.714252,0.930965,0.609553
3000,0.447100,0.315416,0.948764,0.595158,0.508650,0.717121,0.937161,0.618467
3500,0.418500,0.352658,0.960870,0.633239,0.623551,0.643233,0.938368,0.650160
4000,0.433800,0.321023,0.951967,0.615771,0.530920,0.732903,0.943530,0.658607
4500,0.453100,0.300329,0.957090,0.633958,0.574229,0.707556,0.947008,0.673785
5000,0.441100,0.322231,0.961397,0.648284,0.621545,0.677427,0.947714,0.680646


<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tens

TrainOutput(global_step=12000, training_loss=0.4309193318684896, metrics={'train_runtime': 4069.0436, 'train_samples_per_second': 391.406, 'train_steps_per_second': 12.233, 'total_flos': 2.3678144982792e+16, 'train_loss': 0.4309193318684896, 'epoch': 1.2054244098442994})

In [ ]:
print("Final Test Results:")
test_results = trainer.evaluate(eval_dataset=test_dataset)
trainer.log_metrics("test", test_results)
trainer.save_metrics("test", test_results)

# 최종 결과: 훈련 데이터셋에 대한 평가
print("Final Train Results:")
train_results = trainer.evaluate(eval_dataset=train_dataset)
trainer.log_metrics("train", train_results)
trainer.save_metrics("train", train_results)

Final Test Results:


<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


***** test metrics *****
  epoch                   =     1.2054
  eval_accuracy           =     0.9616
  eval_auc-roc            =     0.9584
  eval_f1                 =     0.6701
  eval_loss               =     0.3168
  eval_pr-auc             =      0.728
  eval_precision          =      0.611
  eval_recall             =     0.7418
  eval_runtime            = 0:01:31.15
  eval_samples_per_second =    873.593
  eval_steps_per_second   =     27.305
Final Train Results:


<ipython-input-8-4a337e2e2f44>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


***** train metrics *****
  epoch                   =     1.2054
  eval_accuracy           =     0.9693
  eval_auc-roc            =     0.9729
  eval_f1                 =     0.7366
  eval_loss               =     0.2785
  eval_pr-auc             =     0.7921
  eval_precision          =     0.6695
  eval_recall             =     0.8187
  eval_runtime            = 0:08:30.31
  eval_samples_per_second =    624.189
  eval_steps_per_second   =     19.508


In [ ]:
# from transformers import Trainer, AutoModelForSequenceClassification

# # 저장된 체크포인트 경로
# checkpoint_path = './drive/MyDrive/checkpoints/checkpoint-84000'  # 'checkpoint-xxxx'는 마지막 저장된 체크포인트 디렉토리

# # 모델과 Trainer를 다시 설정하여 학습 이어서 진행
# model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     compute_metrics=compute_metrics
# )

# # 학습 재개
# trainer.train(resume_from_checkpoint=checkpoint_path)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def make_contiguous(model):
    for param in model.parameters():
        param.data = param.data.contiguous()

# 모델의 파라미터를 연속적으로 변환
make_contiguous(model)

# 모델 저장 경로 설정
model_save_path = '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2'  # 원하는 경로로 변경

# 모델 저장
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

('/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2/tokenizer_config.json',
 '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2/special_tokens_map.json',
 '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2/tokenizer.json')

# 제목 수치화

In [3]:
import torch.nn.functional as F
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_save_path = '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2'

# 1. 모델과 토크나이저 불러오기
model = AutoModelForSequenceClassification.from_pretrained(model_save_path)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [4]:
# 기사 제목의 선정성 확률을 계산하는 함수
def predict_title_sensationalism_with_score(title, model, tokenizer, device):
    model.eval()

    # 입력된 기사 제목을 토크나이즈
    inputs = tokenizer(
        title,
        return_tensors="pt",
        truncation=True,
        add_special_tokens=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # 모델에 입력하고 예측 결과 및 logits 얻기
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Softmax를 통해 확률로 변환
        probs = F.softmax(logits, dim=1)
        sensational_prob = probs[0][1].item()  # 선정적인 클래스(1)의 확률

    return sensational_prob

## 제목전처리X

In [11]:
test_df = pd.read_excel('/content/sample_data/크롤링데이터_제목전처리_최종 (1).xlsx')

In [12]:
test_df

,언론사,기사_url,제목,본문
0,노컷뉴스,https://www.nocutnews.co.kr/news/5609089,軍 연이은 성폭력 피해자 사망사건…인권위 직권조사 결정,해군 여성 중사가 남성 상사에게 성추행 피해를 입었다는 신고를 한 후 숙소에서 숨진...
1,노컷뉴스,https://www.nocutnews.co.kr/news/5602233,"'집단감염·성폭력' 軍에 文대통령 ""절치부심하고 심기일전하라""","문재인 대통령은 4일 청해부대 집단감염과 군내 성폭력 사건 등 ""근래 몇 가지 사건..."
2,노컷뉴스,https://www.nocutnews.co.kr/news/5601385,"경북도, 직장 내 성폭력 뿌리뽑는다","경상북도는 ""양성이 평등한 조직 문화를 정착하기 위해 직장 내 성희롱·성폭력 근절 ..."
3,노컷뉴스,https://www.nocutnews.co.kr/news/5602187,"노옥희 울산교육감 ""장애인 성폭력 피해, 책임 통감""","노옥희 울산교육감은 4일 오후 2시 시교육청 프레스센터에서 기자회견을 열어 ""성인장..."
4,노컷뉴스,https://www.nocutnews.co.kr/news/5603779,"연세대 신학교수들, 제자 성폭력 사건 사과 재발방지 노력 약속",연세대 신학 교수들이 전 신학대학원장의 제자 성폭력 사건과 관련해 학교홈페이지 사과...
...,...,...,...,...
13834,프레시안,https://www.pressian.com/pages/articles/202408...,대전시의회 윤리특위 '성추행혐의' 송활섭 의원 제명 의결,대전시의회 윤리특별위원회가 성추행 의혹으로 피소된 송활섭(무소속·대덕구2) 의원에 ...
13835,프레시안,https://www.pressian.com/pages/articles/202408...,여성 공무원성추행시의원 엄정 수사 촉구,충남 천안시청공무원노동조합이 공무원 1만 2512명의 서명을 받아 여성 공무원을 성...
13836,프레시안,https://www.pressian.com/pages/articles/202408...,민주당 충남 여성 당원성추행의혹 시의원 사퇴 촉구,충남도당 여성 당원들이 1일 도당 사무실에서 성명을 발표하고 여성 직원 성추행 의혹...
13837,프레시안,https://www.pressian.com/pages/articles/202408...,성추행 의혹 천안시의원 사과는 없고 때 아닌 공작설…2차 가해 우려,충남 천안시공무원노조가 “천안시의회 강성기 의원이 여성 공무원을 1년 동안 성희롱·...


In [13]:
# '수치' 열을 추가
test_df['선정성 수치'] = 0.0

# 각 제목에 대해 선정성 확률 계산 후 '수치' 열에 추가
for index in range(len(test_df)):
    title = test_df['제목'].iloc[index]  # '새로운 제목' 열에서 제목을 가져옵니다.
    sensational_prob = predict_title_sensationalism_with_score(title, model, tokenizer, device)
    test_df.at[index, '선정성 수치'] = sensational_prob  # 확률 값을 '수치' 열에 추가

In [14]:
sorted_df = test_df.sort_values(by='선정성 수치', ascending=False)
sorted_df

,언론사,기사_url,제목,본문,선정성 수치
7248,데일리안,https://www.dailian.co.kr/news/view/1377326/?s...,"미녀 쫓아가 엉덩이 만지작 男, 사타구니 '퍽' 가격 당했다",일본 후지 뉴스 네트워크(FNN) 등에 따르면 대만 신주시 동구 푸딩리 이장 허즈닝...,0.994534
7135,데일리안,https://www.dailian.co.kr/news/view/1286379/?s...,"""손으로 가슴 만져줘"" 알몸박스女 이곳저곳 출몰하더니 결국",서울 강남구 압구정에 이어 마포구 홍대 일대에서도 알몸에 종이박스만 걸치고 돌아다닌...,0.994380
10161,세계일보,http://www.segye.com/content/html/2022/04/12/2...,"임창정♥서하얀, 노래방서 첫 키스 “청포도 먹여주다…가슴 터질 뻔”",'동상이몽2' 임창정과 서하얀 부부가 노래방에서 한 첫 키스를 했다고 밝혔다.지난 ...,0.994330
12394,세계일보,http://www.segye.com/newsView/20231030508916?O...,‘애마부인’ 안소영 “남동생이 누드사진 찍어줘…목욕할 때 등도 밀어준다”,배우 안소영(67)이 80대 때 연령별 누드 사진을 찍는 게 목표라고 밝히며 50대...,0.994181
5717,데일리안,https://www.dailian.co.kr/news/view/1025456/?s...,"프리미엄 니플패치 방탄꼭지, 핏감 살려주는 그루밍男 필수템",패션과 미용에 아낌없이 투자하는 남자들을 일컫는 신조어를 ‘그루밍족’이라고 한다.뷰...,0.993941
...,...,...,...,...,...
3434,더팩트,http://news.tf.co.kr/read/national/1855316.htm,"광주지역 고3학생, 코로나19 상황에서도 성적 우수",올해 수능을 앞둔 광주지역 고3학생들이 코로나19 상황에서도 우수한 성적을 유지하고...,0.003846
9312,세계일보,http://www.segye.com/content/html/2021/08/30/2...,성희롱 징계받고 취소 소송낸 상태에서 정년퇴직한 공무원,성희롱으로 징계를 받고 취소 소송을 낸 상태에서 정년퇴직한 서울시 공무원이 감봉 3...,0.003837
5663,데일리안,https://www.dailian.co.kr/news/view/1027804/?s...,"학교폭력, 애들 싸움 같습니까?","""안 돼, 돌아가. 바꿔줄 생각 없어. 빨리 돌아가""이제는 유행어가 된 천종호 판사...",0.003834
1467,노컷뉴스,https://www.nocutnews.co.kr/news/5771220,강한 바람에 발사 하루 연기된 누리호,한국형 발사체 누리호의 2차 발사가 하루 미뤄져 오는 16일 이뤄진다. 항공우주연구...,0.003827


In [ ]:
sorted_df = sorted_df.head(100)

In [15]:
sorted_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13839 entries, 7248 to 10867
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   언론사     13839 non-null  object 
 1   기사_url  13839 non-null  object 
 2   제목      13839 non-null  object 
 3   본문      13839 non-null  object 
 4   선정성 수치  13839 non-null  float64
dtypes: float64(1), object(4)
memory usage: 648.7+ KB


In [16]:
sorted_df.to_excel('KcELECTRA_제목전처리O_수치화_전체.xlsx', index=False)

## 제목전처리

In [ ]:
test_df = pd.read_excel('/content/sample_data/크롤링데이터_제목전처리_최종.xlsx')

In [ ]:
# '수치' 열을 추가
test_df['수치'] = 0.0

# 각 제목에 대해 선정성 확률 계산 후 '수치' 열에 추가
for index in range(len(test_df)):
    title = test_df['제목'].iloc[index]  # '제목' 열에서 제목을 가져옵니다.
    sensational_prob = predict_title_sensationalism_with_score(title, model, tokenizer, device)
    test_df.at[index, '수치'] = sensational_prob  # 확률 값을 '수치' 열에 추가

In [ ]:
sorted_df = test_df.sort_values(by='수치', ascending=False)
sorted_df

,언론사,기사_url,제목,본문,수치
7248,데일리안,https://www.dailian.co.kr/news/view/1377326/?s...,"미녀 쫓아가 엉덩이 만지작 男, 사타구니 '퍽' 가격 당했다",일본 후지 뉴스 네트워크(FNN) 등에 따르면 대만 신주시 동구 푸딩리 이장 허즈닝...,0.995354
10161,세계일보,http://www.segye.com/content/html/2022/04/12/2...,"임창정♥서하얀, 노래방서 첫 키스 “청포도 먹여주다…가슴 터질 뻔”",'동상이몽2' 임창정과 서하얀 부부가 노래방에서 한 첫 키스를 했다고 밝혔다.지난 ...,0.994574
12394,세계일보,http://www.segye.com/newsView/20231030508916?O...,‘애마부인’ 안소영 “남동생이 누드사진 찍어줘…목욕할 때 등도 밀어준다”,배우 안소영(67)이 80대 때 연령별 누드 사진을 찍는 게 목표라고 밝히며 50대...,0.993771
7135,데일리안,https://www.dailian.co.kr/news/view/1286379/?s...,"""손으로 가슴 만져줘"" 알몸박스女 이곳저곳 출몰하더니 결국",서울 강남구 압구정에 이어 마포구 홍대 일대에서도 알몸에 종이박스만 걸치고 돌아다닌...,0.993723
7148,데일리안,https://www.dailian.co.kr/news/view/1372338/?s...,껴안고 엉덩이 '주물럭'…女외노자들만 노린 50대 공장장,국내 중소기업에서 근무하는 50대 남성 공장장이 불법 체류자 신분인 외국인 여직원들...,0.993433
...,...,...,...,...,...
10867,세계일보,http://www.segye.com/content/html/2023/05/14/2...,저출산 원인이 경제력 낮은 남자들 결혼 못 해서?,한국의 비혼과 저출산 문제가 남성들의 소득과 영향이 있을 수 있다는 분석이 나왔다....,0.005145
432,노컷뉴스,https://www.nocutnews.co.kr/news/5459326,코로나19로 신음하는 소상공인,4일 오후 서울 중구 명동 지하상가에 임대료 인하를 요구하는 호소문이 붙어 있다. ...,0.005114
2357,노컷뉴스,https://www.nocutnews.co.kr/news/6080854,삼성전자 반도체 임직원도 허리띠 졸라맸다…연봉 동결,삼성전자 반도체 사업을 담당하는 DS(디바이스솔루션) 사업부의 임원들이 실적악화에 ...,0.004956
9312,세계일보,http://www.segye.com/content/html/2021/08/30/2...,성희롱 징계받고 취소 소송낸 상태에서 정년퇴직한 공무원,성희롱으로 징계를 받고 취소 소송을 낸 상태에서 정년퇴직한 서울시 공무원이 감봉 3...,0.004953


In [ ]:
sorted_df = sorted_df.head(100)

In [ ]:
sorted_df.to_excel('제목전처리O_수치화.xlsx', index=False)